# Silver

Builds every Silver table from RAW, in dependency order. Run top to bottom as role **TRANSFORMER** on **TRANSFORMING_WH**.

Each table has three cells: the quirks that apply, the `CREATE OR REPLACE TABLE` cell, and a check cell. The session runs in UTC so Unix timestamps and their dates match the delivery dates. A check is a query that returns the **violating rows**, the same way a dbt test works. The check cell raises if any check returns rows, so the run stops before downstream tables build on a broken one. Paste a failing check's SQL into a cell to see the bad rows.

## Naming
- Primary key `<object>_id`; foreign keys use the same name as the key they reference.
- Timestamps end in `_at` (`TIMESTAMP_TZ`, from Unix seconds), dates in `_on`.
- Booleans start with `is_` / `has_` (`cancel_at_period_end` keeps its Stripe name).
- Money in major units ends in `_amount` (minor units / 100).
- Currency is uppercase.
- Stripe tables keep `delivered_on` (date in `_FILE`), `_file`, `_file_row`, `_loaded_at`.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

FILE_DATE = "TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}'))"


def check(table, checks):
    failed = []
    for name, sql in checks:
        n = session.sql(f"SELECT COUNT(*) FROM ({sql})").collect()[0][0]
        if n:
            failed.append(f"  {name}: {n} rows\n    {sql}")
    assert not failed, f"{table} failed {len(failed)} of {len(checks)} checks:\n" + "\n".join(failed)
    print(f"{table}: {len(checks)} checks passed")


def not_null(t, *cols):
    return f"not_null({', '.join(cols)})", f"SELECT * FROM {t} WHERE " + " OR ".join(f"{c} IS NULL" for c in cols)


def unique(t, *cols):
    c = ", ".join(cols)
    return f"unique({c})", f"SELECT {c} FROM {t} GROUP BY {c} HAVING COUNT(*) > 1"


def accepted(t, col, values, nullable=False):
    v = ", ".join(f"'{x}'" for x in values)
    null = "" if nullable else f" OR {col} IS NULL"
    return f"accepted_values({col})", f"SELECT * FROM {t} WHERE {col} NOT IN ({v}){null}"


def expect(t, name, condition):
    return name, f"SELECT * FROM {t} WHERE NOT ({condition}) OR ({condition}) IS NULL"


def references(t, col, parent, parent_col=None):
    p = parent_col or col
    return (f"relationships({col} -> {parent}.{p})",
            f"SELECT * FROM {t} c WHERE c.{col} IS NOT NULL AND NOT EXISTS (SELECT 1 FROM {parent} p WHERE p.{p} = c.{col})")


def row_count(t, raw_count_sql):
    return "row count matches distinct RAW grain", f"SELECT 1 FROM (SELECT COUNT(*) n FROM {t}) s, ({raw_count_sql}) r WHERE s.n != r.n"


def delivery(t):
    return [
        expect(t, "delivered_on = date in _file", f"delivered_on = {FILE_DATE}"),
        expect(t, "delivered_on in range", "delivered_on BETWEEN '2024-01-01' AND CURRENT_DATE"),
        expect(t, "created_at in range", "created_at BETWEEN '2023-12-01' AND CURRENT_TIMESTAMP"),
        expect(t, "created_at <= delivered_on", "created_at::date <= delivered_on"),
        not_null(t, "_file", "_file_row", "_loaded_at"),
    ]


def stripe_object(t, raw, pk):
    return [
        not_null(t, pk, "delivered_on"),
        unique(t, pk, "delivered_on"),
        row_count(t, f"SELECT COUNT(DISTINCT RAW:id, _file) n FROM {raw}"),
        ("keeps the last row of each delivery",
         f"SELECT * FROM {t} s JOIN (SELECT RAW:id::string id, _file, MAX(_file_row) last_row FROM {raw} GROUP BY 1, 2) r "
         f"ON r.id = s.{pk} AND r._file = s._file WHERE s._file_row != r.last_row"),
        *delivery(t),
    ]


CURRENCIES = ["EUR", "GBP", "USD"]

In [ ]:
%%sql
ALTER SESSION SET TIMEZONE = 'UTC';
CREATE SCHEMA IF NOT EXISTS SILVER.STRIPE;
CREATE SCHEMA IF NOT EXISTS SILVER.SEGMENT;
CREATE SCHEMA IF NOT EXISTS SILVER.ECB;

## SILVER.STRIPE.PRICES
Columns: `price_id`, `product_id`, `currency`, `unit_amount`, `billing_interval`, `nickname`, `is_active`, `created_at`, `delivered_on`, `_file`, `_file_row`, `_loaded_at`

- Full delivery history: one row per price per delivery date.
- A price can appear twice in one file: an exact re-delivery of the previous day's state, then the new state. Keep the last `_FILE_ROW`.
- Annual prices: `billing_interval = 'year'`, `unit_amount` is the yearly price per seat.
- 2025-03-01 price increase: old Pro prices get a new delivery with `active = false`.

In [ ]:
%%sql
CREATE OR REPLACE TABLE SILVER.STRIPE.PRICES AS
SELECT
    RAW:id::string AS price_id,
    RAW:product::string AS product_id,
    UPPER(RAW:currency::string) AS currency,
    (RAW:unit_amount::number / 100)::number(12, 2) AS unit_amount,
    RAW:recurring.interval::string AS billing_interval,
    RAW:nickname::string AS nickname,
    RAW:active::boolean AS is_active,
    TO_TIMESTAMP_TZ(RAW:created::number) AS created_at,
    TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}')) AS delivered_on,
    _file,
    _file_row,
    _loaded_at
FROM RAW.STRIPE.PRICES
QUALIFY ROW_NUMBER() OVER (PARTITION BY RAW:id, _file ORDER BY _file_row DESC) = 1;

In [ ]:
T = "SILVER.STRIPE.PRICES"
check(T, [
    *stripe_object(T, "RAW.STRIPE.PRICES", "price_id"),
    not_null(T, "product_id", "unit_amount", "is_active"),
    accepted(T, "currency", CURRENCIES),
    accepted(T, "billing_interval", ["month", "year"]),
    expect(T, "unit_amount in major units (0 to 999)", "unit_amount >= 0 AND unit_amount < 1000"),
])

## SILVER.STRIPE.PRODUCTS
Columns: `product_id`, `name`, `is_active`, `created_at`, `delivered_on`, `_file`, `_file_row`, `_loaded_at`

In [ ]:
%%sql
CREATE OR REPLACE TABLE SILVER.STRIPE.PRODUCTS AS
SELECT
    RAW:id::string AS product_id,
    RAW:name::string AS name,
    RAW:active::boolean AS is_active,
    TO_TIMESTAMP_TZ(RAW:created::number) AS created_at,
    TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}')) AS delivered_on,
    _file,
    _file_row,
    _loaded_at
FROM RAW.STRIPE.PRODUCTS
QUALIFY ROW_NUMBER() OVER (PARTITION BY RAW:id, _file ORDER BY _file_row DESC) = 1;

In [ ]:
T = "SILVER.STRIPE.PRODUCTS"
check(T, [
    *stripe_object(T, "RAW.STRIPE.PRODUCTS", "product_id"),
    not_null(T, "name"),
])
check("SILVER.STRIPE.PRICES", [references("SILVER.STRIPE.PRICES", "product_id", T)])

## SILVER.STRIPE.COUPONS
Columns: `coupon_id`, `percent_off`, `duration`, `duration_in_months`, `created_at`, `delivered_on`, `_file`, `_file_row`, `_loaded_at`

- `WELCOME20`: 20% for 3 months (`duration = 'repeating'`). `PARTNER10`: 10% forever.

In [ ]:
%%sql
CREATE OR REPLACE TABLE SILVER.STRIPE.COUPONS AS
SELECT
    RAW:id::string AS coupon_id,
    RAW:percent_off::number(5, 2) AS percent_off,
    RAW:duration::string AS duration,
    RAW:duration_in_months::int AS duration_in_months,
    TO_TIMESTAMP_TZ(RAW:created::number) AS created_at,
    TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}')) AS delivered_on,
    _file,
    _file_row,
    _loaded_at
FROM RAW.STRIPE.COUPONS
QUALIFY ROW_NUMBER() OVER (PARTITION BY RAW:id, _file ORDER BY _file_row DESC) = 1;

In [ ]:
T = "SILVER.STRIPE.COUPONS"
check(T, [
    *stripe_object(T, "RAW.STRIPE.COUPONS", "coupon_id"),
    accepted(T, "duration", ["repeating", "forever"]),
    expect(T, "percent_off between 0 and 100", "percent_off > 0 AND percent_off <= 100"),
    expect(T, "duration_in_months set only when repeating", "(duration = 'repeating') = (duration_in_months IS NOT NULL)"),
])

## SILVER.STRIPE.CUSTOMERS
Columns: `customer_id`, `workspace_id`, `name`, `email`, `currency`, `country`, `acquisition_channel`, `is_delinquent`, `created_at`, `delivered_on`, `_file`, `_file_row`, `_loaded_at`

- **Schema change 2025-06-01:** `metadata.acquisition_channel` is renamed `metadata.signup_channel`. `COALESCE` both into `acquisition_channel`.
- `workspace_id` comes from `metadata.workspace_id`; it is the join key to Segment.
- Currency is lowercase in RAW.

In [ ]:
%%sql
CREATE OR REPLACE TABLE SILVER.STRIPE.CUSTOMERS AS
SELECT
    RAW:id::string AS customer_id,
    RAW:metadata.workspace_id::string AS workspace_id,
    RAW:name::string AS name,
    RAW:email::string AS email,
    UPPER(RAW:currency::string) AS currency,
    RAW:address.country::string AS country,
    COALESCE(RAW:metadata.signup_channel::string, RAW:metadata.acquisition_channel::string) AS acquisition_channel,
    RAW:delinquent::boolean AS is_delinquent,
    TO_TIMESTAMP_TZ(RAW:created::number) AS created_at,
    TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}')) AS delivered_on,
    _file,
    _file_row,
    _loaded_at
FROM RAW.STRIPE.CUSTOMERS
QUALIFY ROW_NUMBER() OVER (PARTITION BY RAW:id, _file ORDER BY _file_row DESC) = 1;

In [ ]:
T = "SILVER.STRIPE.CUSTOMERS"
check(T, [
    *stripe_object(T, "RAW.STRIPE.CUSTOMERS", "customer_id"),
    not_null(T, "workspace_id", "is_delinquent"),
    accepted(T, "currency", CURRENCIES),
    accepted(T, "country", ["ES", "GB", "US"]),
    accepted(T, "acquisition_channel", ["organic", "paid_search", "referral", "partner"]),
    ("one workspace per customer", f"SELECT customer_id FROM {T} GROUP BY 1 HAVING COUNT(DISTINCT workspace_id) > 1"),
])

## SILVER.STRIPE.SUBSCRIPTIONS
Columns: `subscription_id`, `customer_id`, `status`, `currency`, `created_at`, `started_at`, `current_period_start_at`, `current_period_end_at`, `trial_start_at`, `trial_end_at`, `cancel_at_period_end`, `canceled_at`, `ended_at`, `cancellation_reason`, `coupon_id`, `discount_start_at`, `discount_end_at`, `delivered_on`, `_file`, `_file_row`, `_loaded_at`

- `cancellation_reason` from `cancellation_details.reason`; discount fields from `discount` (`discount.coupon.id`, `discount.start`, `discount.end`). `discount_end_at` is NULL for forever coupons.
- Items go to their own table below; don't keep `items` here.
- Free to paid is an update to the same subscription.

In [ ]:
%%sql
CREATE OR REPLACE TABLE SILVER.STRIPE.SUBSCRIPTIONS AS
SELECT
    RAW:id::string AS subscription_id,
    RAW:customer::string AS customer_id,
    RAW:status::string AS status,
    UPPER(RAW:currency::string) AS currency,
    TO_TIMESTAMP_TZ(RAW:created::number) AS created_at,
    TO_TIMESTAMP_TZ(RAW:start_date::number) AS started_at,
    TO_TIMESTAMP_TZ(RAW:current_period_start::number) AS current_period_start_at,
    TO_TIMESTAMP_TZ(RAW:current_period_end::number) AS current_period_end_at,
    TO_TIMESTAMP_TZ(RAW:trial_start::number) AS trial_start_at,
    TO_TIMESTAMP_TZ(RAW:trial_end::number) AS trial_end_at,
    RAW:cancel_at_period_end::boolean AS cancel_at_period_end,
    TO_TIMESTAMP_TZ(RAW:canceled_at::number) AS canceled_at,
    TO_TIMESTAMP_TZ(RAW:ended_at::number) AS ended_at,
    RAW:cancellation_details.reason::string AS cancellation_reason,
    RAW:discount.coupon.id::string AS coupon_id,
    TO_TIMESTAMP_TZ(RAW:discount.start::number) AS discount_start_at,
    TO_TIMESTAMP_TZ(RAW:discount.end::number) AS discount_end_at,
    TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}')) AS delivered_on,
    _file,
    _file_row,
    _loaded_at
FROM RAW.STRIPE.SUBSCRIPTIONS
QUALIFY ROW_NUMBER() OVER (PARTITION BY RAW:id, _file ORDER BY _file_row DESC) = 1;

In [ ]:
T = "SILVER.STRIPE.SUBSCRIPTIONS"
check(T, [
    *stripe_object(T, "RAW.STRIPE.SUBSCRIPTIONS", "subscription_id"),
    not_null(T, "customer_id", "status", "current_period_start_at", "current_period_end_at", "cancel_at_period_end"),
    accepted(T, "status", ["trialing", "active", "past_due", "canceled"]),
    accepted(T, "cancellation_reason", ["cancellation_requested", "payment_failed"], nullable=True),
    accepted(T, "currency", CURRENCIES),
    expect(T, "ended_at set only when canceled", "(status = 'canceled') = (ended_at IS NOT NULL)"),
    expect(T, "trial dates set only with a trial", "(trial_start_at IS NULL) = (trial_end_at IS NULL)"),
    expect(T, "period end after period start", "current_period_end_at > current_period_start_at"),
    references(T, "customer_id", "SILVER.STRIPE.CUSTOMERS"),
    references(T, "coupon_id", "SILVER.STRIPE.COUPONS"),
])

## SILVER.STRIPE.SUBSCRIPTION_ITEMS
Columns: `subscription_item_id`, `subscription_id`, `price_id`, `quantity`, `unit_amount`, `billing_interval`, `currency`, `delivered_on`, `_file`, `_file_row`

- `LATERAL FLATTEN` over `items.data`. One row per item per subscription delivery.
- Take `unit_amount` and interval from the price embedded in the item, not from `PRICES`: grandfathered subscriptions keep their old price.
- Flatten only the subscription rows you kept above (same `_file_row`), or a stale same-day duplicate leaks in.

In [ ]:
%%sql
CREATE OR REPLACE TABLE SILVER.STRIPE.SUBSCRIPTION_ITEMS AS
WITH subscriptions AS (
    SELECT * FROM RAW.STRIPE.SUBSCRIPTIONS
    QUALIFY ROW_NUMBER() OVER (PARTITION BY RAW:id, _file ORDER BY _file_row DESC) = 1
)
SELECT
    i.value:id::string AS subscription_item_id,
    s.RAW:id::string AS subscription_id,
    i.value:price.id::string AS price_id,
    i.value:quantity::int AS quantity,
    (i.value:price.unit_amount::number / 100)::number(12, 2) AS unit_amount,
    i.value:price.recurring.interval::string AS billing_interval,
    UPPER(i.value:price.currency::string) AS currency,
    TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}')) AS delivered_on,
    s._file,
    s._file_row
FROM subscriptions s,
    LATERAL FLATTEN(s.RAW:items.data) i;

In [ ]:
T = "SILVER.STRIPE.SUBSCRIPTION_ITEMS"
S = "SILVER.STRIPE.SUBSCRIPTIONS"
check(T, [
    not_null(T, "subscription_item_id", "subscription_id", "price_id", "quantity", "unit_amount", "delivered_on"),
    unique(T, "subscription_item_id", "delivered_on"),
    row_count(T, "SELECT COUNT(DISTINCT i.value:id, s._file) n FROM RAW.STRIPE.SUBSCRIPTIONS s, LATERAL FLATTEN(s.RAW:items.data) i"),
    ("comes from a kept subscription row",
     f"SELECT * FROM {T} i WHERE NOT EXISTS (SELECT 1 FROM {S} s WHERE s.subscription_id = i.subscription_id "
     f"AND s.delivered_on = i.delivered_on AND s._file_row = i._file_row)"),
    ("every subscription row has an item",
     f"SELECT * FROM {S} s WHERE NOT EXISTS (SELECT 1 FROM {T} i WHERE i.subscription_id = s.subscription_id AND i.delivered_on = s.delivered_on)"),
    expect(T, "quantity >= 1", "quantity >= 1"),
    expect(T, "unit_amount in major units (0 to 999)", "unit_amount >= 0 AND unit_amount < 1000"),
    accepted(T, "billing_interval", ["month", "year"]),
    accepted(T, "currency", CURRENCIES),
    references(T, "price_id", "SILVER.STRIPE.PRICES"),
])

## SILVER.STRIPE.INVOICES
Columns: `invoice_id`, `customer_id`, `subscription_id`, `status`, `billing_reason`, `currency`, `subtotal_amount`, `total_amount`, `total_excluding_tax_amount`, `due_amount`, `paid_amount`, `remaining_amount`, `attempt_count`, `created_at`, `period_start_at`, `period_end_at`, `finalized_at`, `paid_at`, `marked_uncollectible_at`, `delivered_on`, `_file`, `_file_row`, `_loaded_at`

- **Schema change 2025-06-01:** `total_excluding_tax` exists only in records delivered from that date. Leave it NULL before.
- Status timestamps come from `status_transitions`.

In [ ]:
%%sql
CREATE OR REPLACE TABLE SILVER.STRIPE.INVOICES AS
SELECT
    RAW:id::string AS invoice_id,
    RAW:customer::string AS customer_id,
    RAW:subscription::string AS subscription_id,
    RAW:status::string AS status,
    RAW:billing_reason::string AS billing_reason,
    UPPER(RAW:currency::string) AS currency,
    (RAW:subtotal::number / 100)::number(12, 2) AS subtotal_amount,
    (RAW:total::number / 100)::number(12, 2) AS total_amount,
    (RAW:total_excluding_tax::number / 100)::number(12, 2) AS total_excluding_tax_amount,
    (RAW:amount_due::number / 100)::number(12, 2) AS due_amount,
    (RAW:amount_paid::number / 100)::number(12, 2) AS paid_amount,
    (RAW:amount_remaining::number / 100)::number(12, 2) AS remaining_amount,
    RAW:attempt_count::int AS attempt_count,
    TO_TIMESTAMP_TZ(RAW:created::number) AS created_at,
    TO_TIMESTAMP_TZ(RAW:period_start::number) AS period_start_at,
    TO_TIMESTAMP_TZ(RAW:period_end::number) AS period_end_at,
    TO_TIMESTAMP_TZ(RAW:status_transitions.finalized_at::number) AS finalized_at,
    TO_TIMESTAMP_TZ(RAW:status_transitions.paid_at::number) AS paid_at,
    TO_TIMESTAMP_TZ(RAW:status_transitions.marked_uncollectible_at::number) AS marked_uncollectible_at,
    TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}')) AS delivered_on,
    _file,
    _file_row,
    _loaded_at
FROM RAW.STRIPE.INVOICES
QUALIFY ROW_NUMBER() OVER (PARTITION BY RAW:id, _file ORDER BY _file_row DESC) = 1;

In [ ]:
T = "SILVER.STRIPE.INVOICES"
check(T, [
    *stripe_object(T, "RAW.STRIPE.INVOICES", "invoice_id"),
    not_null(T, "customer_id", "subscription_id", "status", "billing_reason", "subtotal_amount", "total_amount", "paid_amount"),
    accepted(T, "status", ["paid", "open", "uncollectible"]),
    accepted(T, "billing_reason", ["subscription_create", "subscription_cycle", "subscription_update"]),
    accepted(T, "currency", CURRENCIES),
    expect(T, "amounts >= 0", "subtotal_amount >= 0 AND total_amount >= 0 AND paid_amount >= 0"),
    expect(T, "total <= subtotal (discounts only reduce)", "total_amount <= subtotal_amount"),
    expect(T, "total_excluding_tax present from 2025-06-01 only", "(delivered_on >= '2025-06-01') = (total_excluding_tax_amount IS NOT NULL)"),
    expect(T, "paid invoices have paid_at", "status != 'paid' OR paid_at IS NOT NULL"),
    references(T, "subscription_id", "SILVER.STRIPE.SUBSCRIPTIONS"),
    references(T, "customer_id", "SILVER.STRIPE.CUSTOMERS"),
])

## SILVER.STRIPE.INVOICE_LINES
Columns: `invoice_line_id`, `invoice_id`, `subscription_id`, `price_id`, `line_amount`, `quantity`, `is_proration`, `currency`, `period_start_at`, `period_end_at`, `delivered_on`, `_file`, `_file_row`

- `LATERAL FLATTEN` over `lines.data`, from the invoice rows you kept.
- Prorations: a negative line for unused time on the old price and a positive line for the new one.

In [ ]:
%%sql
CREATE OR REPLACE TABLE SILVER.STRIPE.INVOICE_LINES AS
WITH invoices AS (
    SELECT * FROM RAW.STRIPE.INVOICES
    QUALIFY ROW_NUMBER() OVER (PARTITION BY RAW:id, _file ORDER BY _file_row DESC) = 1
)
SELECT
    l.value:id::string AS invoice_line_id,
    i.RAW:id::string AS invoice_id,
    l.value:subscription::string AS subscription_id,
    l.value:price.id::string AS price_id,
    (l.value:amount::number / 100)::number(12, 2) AS line_amount,
    l.value:quantity::int AS quantity,
    l.value:proration::boolean AS is_proration,
    UPPER(l.value:currency::string) AS currency,
    TO_TIMESTAMP_TZ(l.value:period.start::number) AS period_start_at,
    TO_TIMESTAMP_TZ(l.value:period.end::number) AS period_end_at,
    TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}')) AS delivered_on,
    i._file,
    i._file_row
FROM invoices i,
    LATERAL FLATTEN(i.RAW:lines.data) l;

In [ ]:
T = "SILVER.STRIPE.INVOICE_LINES"
I = "SILVER.STRIPE.INVOICES"
check(T, [
    not_null(T, "invoice_line_id", "invoice_id", "price_id", "line_amount", "is_proration", "delivered_on"),
    unique(T, "invoice_line_id", "delivered_on"),
    row_count(T, "SELECT COUNT(DISTINCT l.value:id, i._file) n FROM RAW.STRIPE.INVOICES i, LATERAL FLATTEN(i.RAW:lines.data) l"),
    ("comes from a kept invoice row",
     f"SELECT * FROM {T} l WHERE NOT EXISTS (SELECT 1 FROM {I} i WHERE i.invoice_id = l.invoice_id "
     f"AND i.delivered_on = l.delivered_on AND i._file_row = l._file_row)"),
    ("lines sum to the invoice subtotal",
     f"SELECT * FROM {I} i LEFT JOIN (SELECT invoice_id, delivered_on, SUM(line_amount) s FROM {T} GROUP BY 1, 2) l "
     f"USING (invoice_id, delivered_on) WHERE l.s IS NULL OR ABS(l.s - i.subtotal_amount) > 0.001"),
    expect(T, "only prorations go negative", "line_amount >= 0 OR is_proration"),
    accepted(T, "currency", CURRENCIES),
    references(T, "price_id", "SILVER.STRIPE.PRICES"),
])

## SILVER.STRIPE.CHARGES
Columns: `charge_id`, `invoice_id`, `customer_id`, `status`, `charge_amount`, `refunded_amount`, `currency`, `failure_code`, `is_paid`, `is_refunded`, `created_at`, `delivered_on`, `_file`, `_file_row`, `_loaded_at`

- Refunds set `refunded = true` and get a new delivery of the charge.

In [ ]:
%%sql
CREATE OR REPLACE TABLE SILVER.STRIPE.CHARGES AS
SELECT
    RAW:id::string AS charge_id,
    RAW:invoice::string AS invoice_id,
    RAW:customer::string AS customer_id,
    RAW:status::string AS status,
    (RAW:amount::number / 100)::number(12, 2) AS charge_amount,
    (RAW:amount_refunded::number / 100)::number(12, 2) AS refunded_amount,
    UPPER(RAW:currency::string) AS currency,
    RAW:failure_code::string AS failure_code,
    RAW:paid::boolean AS is_paid,
    RAW:refunded::boolean AS is_refunded,
    TO_TIMESTAMP_TZ(RAW:created::number) AS created_at,
    TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}')) AS delivered_on,
    _file,
    _file_row,
    _loaded_at
FROM RAW.STRIPE.CHARGES
QUALIFY ROW_NUMBER() OVER (PARTITION BY RAW:id, _file ORDER BY _file_row DESC) = 1;

In [ ]:
T = "SILVER.STRIPE.CHARGES"
check(T, [
    *stripe_object(T, "RAW.STRIPE.CHARGES", "charge_id"),
    not_null(T, "invoice_id", "status", "charge_amount", "is_refunded"),
    accepted(T, "status", ["succeeded", "failed"]),
    accepted(T, "failure_code", ["card_declined", "insufficient_funds", "expired_card"], nullable=True),
    accepted(T, "currency", CURRENCIES),
    expect(T, "failure_code set only on failed charges", "(status = 'failed') = (failure_code IS NOT NULL)"),
    expect(T, "charge_amount > 0", "charge_amount > 0"),
    expect(T, "refunded only if succeeded", "NOT is_refunded OR status = 'succeeded'"),
    references(T, "invoice_id", "SILVER.STRIPE.INVOICES"),
])

## SILVER.STRIPE.REFUNDS
Columns: `refund_id`, `charge_id`, `refund_amount`, `currency`, `reason`, `status`, `created_at`, `delivered_on`, `_file`, `_file_row`, `_loaded_at`

In [ ]:
%%sql
CREATE OR REPLACE TABLE SILVER.STRIPE.REFUNDS AS
SELECT
    RAW:id::string AS refund_id,
    RAW:charge::string AS charge_id,
    (RAW:amount::number / 100)::number(12, 2) AS refund_amount,
    UPPER(RAW:currency::string) AS currency,
    RAW:reason::string AS reason,
    RAW:status::string AS status,
    TO_TIMESTAMP_TZ(RAW:created::number) AS created_at,
    TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}')) AS delivered_on,
    _file,
    _file_row,
    _loaded_at
FROM RAW.STRIPE.REFUNDS
QUALIFY ROW_NUMBER() OVER (PARTITION BY RAW:id, _file ORDER BY _file_row DESC) = 1;

In [ ]:
T = "SILVER.STRIPE.REFUNDS"
check(T, [
    *stripe_object(T, "RAW.STRIPE.REFUNDS", "refund_id"),
    not_null(T, "charge_id", "refund_amount"),
    accepted(T, "currency", CURRENCIES),
    expect(T, "refund_amount > 0", "refund_amount > 0"),
    references(T, "charge_id", "SILVER.STRIPE.CHARGES"),
])

## SILVER.SEGMENT.TRACKS
Columns: `message_id`, `event`, `user_id`, `workspace_id`, `event_at`, `received_at`, `properties`, `_file`, `_file_row`, `_loaded_at`

- About 1% of events appear twice with the same `messageId`: one row per `message_id`.
- `event_at` from `timestamp`, `received_at` from `receivedAt`. Files are partitioned by `receivedAt`, so late events sit in a later file.
- `workspace_id` from `context.groupId`. Keep `properties` as VARIANT.

In [ ]:
%%sql
CREATE OR REPLACE TABLE SILVER.SEGMENT.TRACKS AS
SELECT
    RAW:messageId::string AS message_id,
    RAW:event::string AS event,
    RAW:userId::string AS user_id,
    RAW:context.groupId::string AS workspace_id,
    TO_TIMESTAMP_TZ(RAW:timestamp::string) AS event_at,
    TO_TIMESTAMP_TZ(RAW:receivedAt::string) AS received_at,
    RAW:properties AS properties,
    _file,
    _file_row,
    _loaded_at
FROM RAW.SEGMENT.TRACKS
QUALIFY ROW_NUMBER() OVER (PARTITION BY RAW:messageId ORDER BY _file, _file_row) = 1;

In [ ]:
T = "SILVER.SEGMENT.TRACKS"
check(T, [
    not_null(T, "message_id", "event", "user_id", "workspace_id", "event_at", "received_at", "_file", "_file_row", "_loaded_at"),
    unique(T, "message_id"),
    row_count(T, "SELECT COUNT(DISTINCT RAW:messageId) n FROM RAW.SEGMENT.TRACKS"),
    accepted(T, "event", ["signed_up", "workspace_created", "invited_teammate", "feature_used", "seat_added"]),
    expect(T, "received_at >= event_at", "received_at >= event_at"),
    expect(T, "received_at date = date in _file", f"TO_DATE(CONVERT_TIMEZONE('UTC', received_at)) = {FILE_DATE}"),
    expect(T, "event_at in range", "event_at BETWEEN '2024-01-01' AND CURRENT_TIMESTAMP"),
    references(T, "workspace_id", "SILVER.STRIPE.CUSTOMERS"),
])

## SILVER.ECB.RATES
Columns: `currency`, `rate_on`, `rate`

- Units of currency per 1 EUR. GBP and USD only; EUR is the base.
- Daily extracts overlap by 10 days: one row per currency and date.
- `OBS_VALUE` is a string in RAW.

In [ ]:
%%sql
CREATE OR REPLACE TABLE SILVER.ECB.RATES AS
SELECT
    RAW:CURRENCY::string AS currency,
    RAW:TIME_PERIOD::date AS rate_on,
    RAW:OBS_VALUE::number(18, 6) AS rate
FROM RAW.ECB.RATES
QUALIFY ROW_NUMBER() OVER (PARTITION BY RAW:CURRENCY, RAW:TIME_PERIOD ORDER BY _file DESC, _file_row DESC) = 1;

In [ ]:
T = "SILVER.ECB.RATES"
check(T, [
    not_null(T, "currency", "rate_on", "rate"),
    unique(T, "currency", "rate_on"),
    row_count(T, "SELECT COUNT(DISTINCT RAW:CURRENCY, RAW:TIME_PERIOD) n FROM RAW.ECB.RATES"),
    accepted(T, "currency", ["GBP", "USD"]),
    expect(T, "rate > 0", "rate > 0"),
    expect(T, "business days only", "DAYOFWEEKISO(rate_on) <= 5"),
    expect(T, "rate_on in range", "rate_on BETWEEN '2023-12-01' AND CURRENT_DATE"),
])

## Summary

In [ ]:
%%sql
SELECT table_schema, table_name, row_count
FROM SILVER.INFORMATION_SCHEMA.TABLES
WHERE table_schema IN ('STRIPE', 'SEGMENT', 'ECB')
ORDER BY 1, 2;